# 03. Training & Evaluation

**Phase 4**: 모든 오염 데이터셋 × 5개 모델 학습 & 평가

**사전 조건**: `01`, `02` 노트북 실행 완료

---

## 0. 환경 설정

In [ ]:
# ============================================================
# 0-1. Google Drive 마운트 & 의존성
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

%pip install -q xgboost

import os, glob
import pandas as pd
import numpy as np
from time import time
import warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/capstone/dsc'
RAW_DIR = f'{BASE}/data/raw'
POLLUTED_DIR = f'{BASE}/data/polluted'
RESULTS_DIR = f'{BASE}/results'

print('환경 설정 완료')

In [ ]:
# ============================================================
# 0-2. 데이터셋 메타데이터 & 모델 정의
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

DATASETS = {
    'TelcoCustomerChurn': {
        'target': 'Churn',
        'numerical_cols': ['tenure', 'MonthlyCharges', 'TotalCharges'],
        'categorical_cols': [
            'gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaperlessBilling', 'PaymentMethod'
        ],
    },
    'SouthGermanCredit': {
        'target': 'credit_risk',
        'numerical_cols': ['duration', 'amount', 'age'],
        'categorical_cols': [
            'status', 'credit_history', 'purpose', 'savings',
            'employment_duration', 'installment_rate', 'personal_status_sex',
            'other_debtors', 'present_residence', 'property',
            'other_installment_plans', 'housing', 'number_credits',
            'job', 'people_liable', 'telephone', 'foreign_worker'
        ],
    },
    'letter': {
        'target': 'lettr',
        'numerical_cols': [
            'x-box', 'y-box', 'width', 'high', 'onpix', 'x-bar',
            'y-bar', 'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br',
            'x-ege', 'xegvy', 'y-ege', 'yegvx'
        ],
        'categorical_cols': [],
    },
}


def get_models():
    return {
        'LogisticRegression': LogisticRegression(
            solver='lbfgs', max_iter=2000, random_state=42, n_jobs=-1
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1
        ),
        'XGBoost': XGBClassifier(
            n_estimators=100, random_state=42, n_jobs=-1,
            use_label_encoder=False, eval_metric='mlogloss'
        ),
        'SVC': SVC(
            kernel='linear', random_state=42, probability=True
        ),
        'MLP': MLPClassifier(
            hidden_layer_sizes=(100, 100, 100, 100, 100),
            random_state=42, max_iter=1000
        ),
    }


def prepare_data(df, meta):
    """전처리: train/test 분할 + 인코딩 + 스케일링."""
    target = meta['target']
    num_cols = [c for c in meta['numerical_cols'] if c in df.columns]
    cat_cols = [c for c in meta['categorical_cols'] if c in df.columns]
    feature_cols = num_cols + cat_cols

    X = df[feature_cols].copy()
    y = df[target].copy()

    for col in num_cols:
        X[col] = pd.to_numeric(X[col], errors='coerce')
    for col in num_cols:
        X[col] = X[col].fillna(X[col].median() if X[col].notna().any() else 0)
    for col in cat_cols:
        X[col] = X[col].fillna('MISSING').astype(str)

    le = LabelEncoder()
    y = le.fit_transform(y.astype(str))

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=1, stratify=y
    )

    transformers = []
    if num_cols:
        transformers.append(('num', StandardScaler(), num_cols))
    if cat_cols:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols))

    preprocessor = ColumnTransformer(transformers, remainder='drop')
    X_train_t = preprocessor.fit_transform(X_train)
    X_test_t = preprocessor.transform(X_test)

    n_classes = len(np.unique(y))
    return X_train_t, X_test_t, y_train, y_test, n_classes


def evaluate_model(model, X_test, y_test, n_classes):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    try:
        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_test)
        else:
            y_prob = model.decision_function(X_test)
        if n_classes == 2:
            if y_prob.ndim == 2:
                y_prob = y_prob[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        else:
            auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    except Exception:
        auc = np.nan
    return {'accuracy': round(acc, 4), 'f1_macro': round(f1, 4), 'auc_roc': round(auc, 4)}


print('모델 및 유틸리티 정의 완료')

## 1. 오염 데이터 목록 스캔

In [ ]:
# ============================================================
# 1-1. polluted/ 디렉토리 스캔하여 실험 목록 구성
# ============================================================
experiments = []

for ds_name in DATASETS:
    ds_dir = f'{POLLUTED_DIR}/{ds_name}'
    if not os.path.isdir(ds_dir):
        continue
    for folder in sorted(os.listdir(ds_dir)):
        data_csv = f'{ds_dir}/{folder}/data.csv'
        if not os.path.isfile(data_csv):
            continue
        # folder 형식: polluter_name_level  (e.g. completeness_25)
        parts = folder.rsplit('_', 1)
        polluter_name = parts[0]
        level = int(parts[1]) / 100 if len(parts) == 2 else 0.0
        experiments.append({
            'dataset': ds_name,
            'polluter': polluter_name,
            'level': level,
            'path': data_csv,
        })

print(f'오염 데이터 {len(experiments)}건 발견')
print(f'모델 5개 × {len(experiments)}건 = 총 {5 * len(experiments)}회 학습 예정')

## 2. 전체 모델 학습 & 평가

In [ ]:
# ============================================================
# 2-1. 학습 루프 (체크포인트 지원)
# ============================================================
perf_path = f'{RESULTS_DIR}/model_performance.csv'

# 기존 결과 로드 (이어서 실행 가능)
if os.path.isfile(perf_path):
    df_perf = pd.read_csv(perf_path)
    existing_keys = set(
        df_perf.apply(lambda r: f"{r['dataset']}|{r['polluter']}|{r['level']}|{r['model']}", axis=1)
    )
    perf_rows = df_perf.to_dict('records')
    print(f'기존 결과 {len(perf_rows)}건 로드')
else:
    existing_keys = set()
    perf_rows = []

total_start = time()
error_log = []
completed = 0
skipped = 0
total_experiments = len(experiments) * 5

for i, exp in enumerate(experiments):
    ds_name = exp['dataset']
    meta = DATASETS[ds_name]

    # 데이터 로드 & 전처리
    try:
        df = pd.read_csv(exp['path'])
        X_train, X_test, y_train, y_test, n_classes = prepare_data(df, meta)
    except Exception as e:
        label = f"{ds_name}/{exp['polluter']}_{int(exp['level']*100)}%"
        error_log.append({'label': label, 'model': 'ALL', 'error': str(e)})
        print(f'  [{i+1}/{len(experiments)}] {label} → 전처리 실패: {e}')
        continue

    for model_name, model in get_models().items():
        key = f"{ds_name}|{exp['polluter']}|{exp['level']}|{model_name}"
        if key in existing_keys:
            skipped += 1
            continue

        try:
            t0 = time()
            model.fit(X_train, y_train)
            scores = evaluate_model(model, X_test, y_test, n_classes)
            elapsed = time() - t0

            row = {
                'dataset': ds_name,
                'polluter': exp['polluter'],
                'level': exp['level'],
                'model': model_name,
                **scores,
            }
            perf_rows.append(row)
            existing_keys.add(key)
            completed += 1

        except Exception as e:
            label = f"{ds_name}/{exp['polluter']}_{int(exp['level']*100)}%/{model_name}"
            error_log.append({'label': label, 'error': str(e)})

    # 진행상황 (데이터셋 단위)
    if (i + 1) % 5 == 0 or i == len(experiments) - 1:
        elapsed_total = time() - total_start
        print(f'  [{i+1}/{len(experiments)}] 완료={completed}, 스킵={skipped}, 에러={len(error_log)}  ({elapsed_total:.0f}s)')

    # 매 10건마다 중간 저장 (Colab 세션 끊김 대비)
    if (i + 1) % 10 == 0:
        pd.DataFrame(perf_rows).to_csv(perf_path, index=False)

# 최종 저장
df_perf = pd.DataFrame(perf_rows)
df_perf.to_csv(perf_path, index=False)

total_elapsed = time() - total_start
print(f'\n=== 학습 완료 ===')
print(f'총 {len(df_perf)}건 (신규 {completed}, 스킵 {skipped}, 에러 {len(error_log)})')
print(f'소요 시간: {total_elapsed:.0f}초 ({total_elapsed/60:.1f}분)')
print(f'저장: {perf_path}')

In [ ]:
# ============================================================
# 2-2. 에러 로그 확인
# ============================================================
if error_log:
    print(f'에러 {len(error_log)}건:')
    for e in error_log:
        print(f'  {e["label"]}: {e.get("error", "unknown")}')
else:
    print('에러 없음')

In [ ]:
# ============================================================
# 2-3. 결과 미리보기
# ============================================================
print(f'\n결과 shape: {df_perf.shape}')
print(f'\n데이터셋별 건수:')
print(df_perf.groupby('dataset').size())
print(f'\npolluter별 건수:')
print(df_perf.groupby('polluter').size())
print(f'\n--- 노트북 03 완료 ---')
print(f'다음: 04_scoreboard.ipynb 실행')

df_perf.head(10)

In [ ]:
# ============================================================
# 3. 실행 로그 저장
# ============================================================
from datetime import datetime

log_lines = []
log_lines.append('# 노트북 03 실행 로그: 모델 학습 & 평가')
log_lines.append('')
log_lines.append(f'- **실행 시각**: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
log_lines.append(f'- **총 실험**: {len(df_perf)}건')
log_lines.append(f'- **신규 학습**: {completed}건')
log_lines.append(f'- **스킵 (기존 결과)**: {skipped}건')
log_lines.append(f'- **에러**: {len(error_log)}건')
log_lines.append(f'- **소요 시간**: {total_elapsed:.0f}초 ({total_elapsed/60:.1f}분)')
log_lines.append('')

# 모델 설정
log_lines.append('## 1. 모델 설정')
log_lines.append('')
log_lines.append('| 모델 | 계열 | 주요 하이퍼파라미터 |')
log_lines.append('|---|---|---|')
log_lines.append('| LogisticRegression | 선형 | solver=lbfgs, max_iter=2000 |')
log_lines.append('| RandomForest | 배깅 | n_estimators=100 |')
log_lines.append('| XGBoost | 부스팅 | n_estimators=100 |')
log_lines.append('| SVC | 커널 | kernel=linear |')
log_lines.append('| MLP | 신경망 | hidden_layers=(100,100,100,100,100), max_iter=1000 |')
log_lines.append('')
log_lines.append('- 전처리: StandardScaler(수치) + OneHotEncoder(범주)')
log_lines.append('- 분할: train 80% / test 20% (stratified)')
log_lines.append('- 평가: Accuracy, F1(macro), AUC-ROC')
log_lines.append('')

# 데이터셋별 결과 요약
log_lines.append('## 2. 데이터셋별 결과 요약')
log_lines.append('')
for ds_name in df_perf['dataset'].unique():
    ds_df = df_perf[df_perf['dataset'] == ds_name]
    log_lines.append(f'### {ds_name} ({len(ds_df)}건)')
    log_lines.append('')
    log_lines.append('| 오염 유형 | 강도 | 모델 | F1(macro) | Accuracy | AUC-ROC |')
    log_lines.append('|---|---|---|---|---|---|')
    for _, row in ds_df.iterrows():
        log_lines.append(f'| {row["polluter"]} | {row["level"]} | {row["model"]} | {row["f1_macro"]} | {row["accuracy"]} | {row["auc_roc"]} |')
    log_lines.append('')

# 모델별 평균 성능
log_lines.append('## 3. 모델별 평균 F1 (전체)')
log_lines.append('')
log_lines.append('| 모델 | 평균 F1 | 최소 F1 | 최대 F1 | 건수 |')
log_lines.append('|---|---|---|---|---|')
model_stats = df_perf.groupby('model')['f1_macro'].agg(['mean', 'min', 'max', 'count'])
for model_name, stats in model_stats.iterrows():
    log_lines.append(f'| {model_name} | {stats["mean"]:.4f} | {stats["min"]:.4f} | {stats["max"]:.4f} | {int(stats["count"])} |')
log_lines.append('')

# 에러
if error_log:
    log_lines.append('## 4. 에러 목록')
    log_lines.append('')
    for e in error_log:
        err_msg = str(e.get('error', 'unknown'))
        if len(err_msg) > 200:
            err_msg = err_msg[:200] + '...'
        log_lines.append(f'- **{e.get("label", "?")}**: {err_msg}')
    log_lines.append('')

# 산출물
log_lines.append('## 5. 산출물')
log_lines.append('')
log_lines.append(f'- `model_performance.csv` — 모델 성능 {len(df_perf)}건')
log_lines.append(f'- `03_execution_log.md` — 이 로그 파일')
log_lines.append('')
log_lines.append('---')
log_lines.append('*이 로그는 노트북 03 실행 시 자동 생성됨*')

log_path = f'{RESULTS_DIR}/03_execution_log.md'
with open(log_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines))
print(f'실행 로그 저장: {log_path}')
